# PHA - Data Science Agent Demo

This notebook demonstrates how to use the **Data Science Agent** from PHA (Personal Health Insights Agent Team).

The Data Science Agent can:
- Analyze your Fitbit/health data
- Generate analysis approaches from natural language questions
- Write and execute Python code to answer your questions
- Provide user-friendly summaries of the results

## Setup

First, let's install the required dependencies and set up our environment.

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install pandas numpy jinja2 google-genai openai anthropic

In [ ]:
import os
import sys

# --- Configuration ---
# Set your API key and provider here
API_KEY = "your-api-key-here"
# Provider options: "gemini", "openai", "anthropic"
PROVIDER = "gemini"


## Load Configuration and Data

We'll use the Settings class to configure paths and API keys.

In [ ]:
from config import Settings
from pha.utils import load_persona

# Create settings (uses environment variables for API keys)
settings = Settings(
    data_dir='../data/sample',
)

print(f"Data directory: {settings.data_dir}")

In [ ]:
# Load the sample data
summary_df, activities_df, profile_df, population_df = load_persona(settings=settings)

print(f"Summary data: {len(summary_df)} days")
print(f"Activities: {len(activities_df)} records")
print(f"\nSummary columns: {list(summary_df.columns)}")

In [ ]:
# Preview the data
print("=== Summary DataFrame (daily metrics) ===")
summary_df.head()

In [ ]:
print("=== Profile DataFrame ===")
profile_df

## Initialize the Data Science Agent

Now let's create the Data Science Agent with our settings.

In [ ]:
from pha.agents import DataScienceAgent

# Create the agent
# Implicitly uses global configuration for LLM backend
agent = DataScienceAgent(settings=settings)
agent.configure(api_key=API_KEY, provider=PROVIDER)  # Explicit config

print("Data Science Agent initialized!")

## Ask Questions About Your Data

Now we can ask natural language questions about the health data!

In [ ]:
# Example 1: Simple statistics query
question = "What is my average daily step count over the last 30 days?"

print(f"Question: {question}")
print("\nProcessing...")

response = agent.query(question)
print(f"\nResponse:\n{response}")

In [ ]:
# Example 2: Sleep analysis
question = "How does my deep sleep compare to light sleep? Show me the breakdown."

print(f"Question: {question}")
print("\nProcessing...")

response = agent.query(question)
print(f"\nResponse:\n{response}")

In [ ]:
# Example 3: Correlation analysis
question = "Is there a relationship between my sleep duration and my resting heart rate?"

print(f"Question: {question}")
print("\nProcessing...")

response = agent.query(question)
print(f"\nResponse:\n{response}")

## Detailed Query (See All Steps)

For debugging or understanding how the agent works, you can use `query_with_details()` to see all intermediate steps.

In [ ]:
question = "What's my most active day of the week based on steps?"

print(f"Question: {question}")
print("\nProcessing with details...")

result = agent.query_with_details(question)

In [ ]:
# View the generated approach
print("=== APPROACH ===")
print(result['approach'])

In [ ]:
# View the generated code
print("=== GENERATED CODE ===")
print(result['code'])

In [ ]:
# View execution results
print("=== EXECUTION RESULT ===")
print(f"Status: {result['execution_result']['status']}")
print(f"Output:\n{result['execution_result']['output']}")
if result['execution_result']['stderr']:
    print(f"Errors:\n{result['execution_result']['stderr']}")

In [ ]:
# View final response
print("=== FINAL RESPONSE ===")
print(result['response'])

## Step-by-Step Execution

You can also run each step individually for more control.

In [ ]:
# Step 1: Generate approach
question = "Am I getting enough sleep compared to the general population?"
approach = agent.generate_approach(question)

print("=== APPROACH ===")
print(approach)

In [ ]:
# Step 2: Generate code
code = agent.generate_code(question, approach)

print("=== CODE ===")
print(code)

In [ ]:
# Step 3: Execute code
result = agent.execute_code(code)

print(f"Status: {result.execution_result.status}")
print(f"\nOutput:\n{result.execution_result.output}")

In [ ]:
# Step 4: Interpret results
interpretation = agent.interpret_results(
    question=question,
    approach=approach,
    code=code,
    execution_results=result.execution_result.output,
)

print("=== INTERPRETATION ===")
print(interpretation)

## Using Different LLM Backends

PHA supports multiple LLM backends via the unified configuration. Here's how to switch between them:

In [ ]:
from pha.llm import configure_global

# Switch to OpenAI
# configure_global(provider='openai', api_key='your-openai-key')
# agent.configure()  # Reconfigure agent to use new backend

# Switch to Anthropic Claude
# configure_global(provider='anthropic', api_key='your-anthropic-key')
# agent.configure()

## Try Your Own Questions!

Now try asking your own questions about the health data:

In [ ]:
# Try your own question!
your_question = "What trends do you see in my heart rate variability?"

response = agent.query(your_question)
print(response)